# Bioreactor Modeling: Governing Equations and Design Basis

In this tutorial, we will learn how to design and model bioreactors for anaerobic, aerobic, and CO2/H2 gas-fed fermentations. We will first introduce the governing phenomenological equations (i.e., mass and energy balances, heat and mass transfer), then move into the following case studies, ordered by increasing complexity:

* **Anaerobic fermentation:** conventional corn ethanol production.
* **Aerobic fermentation:** microbial oil production from sugar.
* **CO2/H2 gas-fed fermentation:** acetic acid production from blast furnace steelmaking flue gas.

## Governing phenomenological equations

There are 4 governing phenomenological equations that need to be satisfied in a bioreactor:

**Mass balance:** 
$∑_{i\mathrm{\ \in\ products}} F_i - ∑_{i\mathrm{\ \in\ feeds}} F_i = ∑_i R_i$

**Energy balance:** 
$\mathrm{Cooling\ Duty} = ∑_{i\mathrm{\ \in\ products}} H_i - ∑_{i\mathrm{\ \in\ feeds}} H_i$

**Mass transfer:** 
$\mathrm{Gas\ Uptake\ Rate = Gas\ Transfer\ Rate}$

**Heat transfer:** 
$\mathrm{Cooling\ Duty = Heat\ Transfer\ Rate}$    

* **$F$** - Molar flow rate of an arbitrary chemical.
* **$R$** - Generation rate.
* **$H$** - Enthalpy flow rate.

For a specific fermentation performance (e.g., titer and yield), we can solve for our mass and energy balances. This leaves us with the heat and mass transfer requirements, which must be satisfied by the detailed design of the bioreactor and its components (e.g., vessel, heat exchanger, compressor, agitator). Alternatively, given a specified design (e.g., vessel aspect ratio, agitation, gas flow rate), we could let the gas mass transfer rate dictate the fermentation performance. 

In the following examples we will assume an experimentally verified fermentation performance and solve for the design of the bioreactor. As an exercise, we will first solve the problem manually with the help of BioSTEAM's thermodynamic engine. Then, we will verify our results using BioSTEAM Unit Operations.

## Corn Ethanol Anaerobic bioreactor

In the conventional dry-grind process for corn ethanol production, the corn is first liquified and saccharified to create a slurry. This glucose is then fermented to ethanol and carbon dioxide using yeast:

$\mathrm{Glucose \rightarrow 2\ Ethanol + 2\ CO_2}$

We will assume the following fermentation performance:

* Feed flow rate: 150,000 kg$\cdot$h$^{-1}$ 
* Feed glucose concentration: 20% wt.
* Ethanol yield: 95% theoretical.
* Temperture: 32 $\degree$C

In [1]:
feed_flow = 150000
x_Glucose = 0.20
yield_EtOH = 0.95
T_operation = 32 + 273.15

### Mass balance

We can solve for the total product flow rates by reacting the feed assuming the given yield. Mathematically, this translates to:

$ F_{i,\ \mathrm{out}} = F_{i,\ \mathrm{in}} + C_i \cdot X_{\mathrm{Glucose}} \cdot F_{\mathrm{Glucose,\ in}} $

* $X$ - Extent of reaction.
* $C$ - Stoichiometric reaction coefficient.
* $i$ - Index for an arbitrary chemical.

We can model the stoichiometric reaction using [Reaction](../API/thermosteam/Reaction.txt) objects to "react" a stream. 

To compute the flow rate of chemicals as they partition into the gas and liquid phases (i.e., the vent and effluent streams), we need to meet the vapor-liquid equilibrium criteria, which translates to the fugacity of each chemical in each phase must be the same. 

$ f^\alpha_i = f^\beta_i$

- $f^\alpha$ - Fugacity in phase $\alpha$ [Pa].
- $f^\beta$ - Fugacity in phase $\beta$  [Pa].

Phase equilibrium can be challenging to solve given how fugacity is a highly nonlinear and tightly coupled function of composition, temperature, and pressure. In BioSTEAM, we can quickly perform vapor-liquid equilibrium to partition the flows into the vent and effluent streams:

In [2]:
import biosteam as bst; bst.nbtutorial()

# Define chemicals
bst.settings.set_thermo(
    ['Water', 'Ethanol', 'Glucose', 'CO2'], 
    db='BioSTEAM'
)

# Create the feed
feed = bst.Stream(
    Water=1 - x_Glucose, 
    Glucose=x_Glucose,
    total_flow=feed_flow,
    units='kg/hr',
    T=T_operation,
)

# Define the reaction
fermentation = bst.Reaction(
    'Glucose -> 2 Ethanol + 2 CO2',
    X=yield_EtOH,
)

# Convert the feed to product
product = feed.copy(ID='') # Infer ID
fermentation(product)

# Perform vapor-liquid equilibrium
product.vle(T=T_operation, P=101325)

feed.show('cwt')
print()
fermentation.show()
print()
product.show('cwt')

Stream: feed
phase: 'l', T: 305.15 K, P: 101325 Pa
composition (%): Water    80
                 Glucose  20
                 -------  1.5e+05 kg/hr

Reaction (by mol):
stoichiometry                 reactant    X[%]
Glucose -> 2 Ethanol + 2 CO2  Glucose    95.00

MultiStream: product
phases: ('g', 'l'), T: 305.15 K, P: 101325 Pa
composition (%): (g) Water    1.87
                     Ethanol  2.15
                     CO2      96
                     -------  1.05e+04 kg/hr
                 (l) Water    85.9
                     Ethanol  10.3
                     Glucose  1.07
                     CO2      2.78
                     -------  1.4e+05 kg/hr


### Energy balance

Now that we have solved the mass balance, we can estimate our cooling duty. The enthalpy flow rate of a stream depends on the temperature, pressure, and material flow rates. In BioSTEAM, we can estimate the enthalpy of a stream by using the [Stream.H](https://biosteam.readthedocs.io/en/latest/API/thermosteam/Stream.html#thermosteam.Stream.H) and [Stream.Hf](https://biosteam.readthedocs.io/en/latest/API/thermosteam/Stream.html#thermosteam.Stream.Hf) properties. Note that `H` does include heats of formation at its reference state while `Hf` is just the heats of formation.

In [3]:
cooling_duty = (product.Hf + product.H) - (feed.Hf + feed.H) # Enthalpy change (including heats of formation)
print('cooling duty:', f'{cooling_duty:.3g}', 'kJ/hr')

cooling duty: -1.43e+07 kJ/hr


### Bioreactor sizing

To size the bioreactor, we will need to make the following heurstic design specifications:

* $R$ - length to diameter: 3
* $V_{\mathrm{max}}$ - maximum reactor volume: 1000 m3
* $f$ - working volume: 95%
* $\tau_{\mathrm{reaction}}$ - reaction time: 60 h.
* $\tau_{\mathrm{cleaning}}$ - clean up time: 3 h
* $\tau_{\mathrm{loading}}$ - loading time: 1 h
* $R$ - height to diameter ratio: 3

This allows us to solve for the reactor volume, number of reactors, and reactor dimensions:

$ V_{\mathrm{total}} = \nu(\tau_{\mathrm{reaction}} + \tau_{\mathrm{cleaning}} + \tau_{\mathrm{loading}}) \cdot f^{-1} $

$ N_{\mathrm{reactors}} = \mathrm{ceil}(V_{\mathrm{total}} \cdot V_{\mathrm{max}^{-1}}) $

$ V_{\mathrm{reactor}} = V_{\mathrm{total}} N_{\mathrm{reactors}}^{-1} $

$ D = (4 \pi  V_{\mathrm{reactor}} R)^{1/3} $

$ H = D \cdot R $

* $\nu$ - Volumetric flow rate.
* $N_{reactors}$ - Number of reactors.
* $V_{total}$ - Total volume of all reactors.
* $V_{reactor}$ - Volume of a reactor.
* $D$ - Diameter.
* $H$ - Height.


In [4]:
from math import ceil, pi as π

# Heuristic specifications
R = 3
V_max = 1000
f = 0.90
tau_reaction = 60
tau_cleanup = 3
tau_loading = 1

# Solve for number of reactors and the volume of an individual reactor 
𝜈 = product['l'].F_vol
V_total = 𝜈 * (tau_reaction + tau_cleanup + tau_loading) / f
N_reactors = ceil(V_total / V_max)
V_reactor = V_total / N_reactors
print('Number of reactors:', N_reactors)
print('Reactor volume:', f"{V_reactor:.3g}", 'm3')

# Solve for the diameter and height
D = (4 * V_reactor / π / R)**(1/3)
H = D * R
print('Reactor height:', f"{H:.3g}", 'm')
print('Reactor diameter:', f"{D:.3g}", 'm')

Number of reactors: 11
Reactor volume: 931 m3
Reactor height: 22 m
Reactor diameter: 7.34 m


## Heat exchange

The most commonly used bioreactor heat exchanger configuration is the jacketed vessel. A full jacket provides a constant heat transfer area to the medium. In order to meet the cooling requirement, the flow rate of the cooling agent (e.g., chilled water) can be varied. The greater the flow rate, the greater the overall heat transfer coefficient, and the greater the driving force (i.e., the temperature difference across the jacket). Because the temperature of the utility changes across the jacket, we use the log-mean temperature difference as the driving force (LMTD).

$ Q = U A \Delta T_{\mathrm{LMTD}} $

$ \Delta T_{\mathrm{LMTD}} = (\Delta T_{\mathrm{in}}-\Delta T_{\mathrm{out}}) \ln(\Delta T_{\mathrm{in}} / \Delta T_{\mathrm{out}})^{-1} $

$ \Delta T_{\mathrm{in}} = (T_{\mathrm{utility,\ in}} - T_{\mathrm{operation}}) $

$ \Delta T_{\mathrm{out}} = (T_{\mathrm{utility,\ out}} - T_{\mathrm{operation}}) $

* $U$ - Overall heat trasfer coefficient.
* $Q$ - Cooling duty.
* $T$ - Temperature.

Estimating the overall heat transfer coefficient can be challenging due to its dependence on detailed design of the bioreactor (e.g., agitation) and mixture properties (e.g., density, conductivity). As a preliminary estimate, we can assume that the temperature of the outlet utility is simply the maximum allowable temperature for smooth regeneration (chilled water cannot come back too hot to the cooler). Because the cost of chilled water usage is estimated based on the total heat transfered (not on the actual flow rate), this assumption does not impact our economic analysis. 

$ F_{\mathrm{utility}} = Q(h_\mathrm{utility,\ in} - h_\mathrm{utility,\ out})^{-1} $ 

* $T_\mathrm{utility,\ out}$ - Outlet utility temperature: 27.22 $\deg$C.
* $T_\mathrm{utility,\ in}$ - Inlet utility temperature: 7.22 $\deg$C.
* $h$ - Specific molar enthalpy.

In [5]:
# Additional assumptions/specifications
T_utility_out = 27.22 + 273.15
T_utility_in = 7.22 + 273.15

# Solve for utility requirements
chilled_water_in = bst.Stream(Water=1, T=T_utility_in)
chilled_water_out = bst.Stream(Water=1, T=T_utility_out)
F_utility = cooling_duty / (chilled_water_in.h - chilled_water_out.h)

print('Chilled water flow rate:',  f'{F_utility:.3g}', 'kmol/hr')

Chilled water flow rate: 9.5e+03 kmol/hr


### Standard Anaerobic Bioreactor Model

BioSTEAM streamlines the modeling, design, costing of bioreactors. Let's use BioSTEAM's AnaerobicBioreactor model to model the corn ethanol fermentation:

In [6]:
import biosteam as bst; bst.nbtutorial()

bioreactor = bst.AnaerobicBioreactor(
    ins=feed, 
    outs=('vent', 'effluent'),
    reactions=fermentation,
    T=T_operation,
    V_wf=f,
    V_max=V_max,
    tau=tau_reaction,
    tau_0=tau_cleanup,
    loading_time=tau_loading,
    length_to_diameter=R,
    heat_exchanger_configuration='jacketed',
)
bioreactor.simulate()
bioreactor.show('cwt')

AnaerobicBioreactor: bioreactor
ins...
[0] feed  
    phase: 'l', T: 305.15 K, P: 101325 Pa
    composition (%): Water    80
                     Glucose  20
                     -------  1.5e+05 kg/hr
outs...
[0] vent  
    phase: 'g', T: 305.15 K, P: 101325 Pa
    composition (%): Water    1.87
                     Ethanol  2.15
                     CO2      96
                     -------  1.05e+04 kg/hr
[1] effluent  
    phase: 'l', T: 305.15 K, P: 101325 Pa
    composition (%): Water    85.9
                     Ethanol  10.3
                     Glucose  1.07
                     CO2      2.78
                     -------  1.4e+05 kg/hr


In [7]:
bioreactor.results(basis='SI')

Anaerobic bioreactor                                            Units           bioreactor
Electricity              Power                                     kW             2.67e+03
                         Cost                                  USD/hr                  209
Chilled water            Duty                                   kJ/hr            -1.43e+07
                         Flow                                 kmol/hr              9.6e+03
                         Cost                                  USD/hr                 71.6
Design                   Reactor volume                            m³                  913
                         Batch time                                hr                   64
                         Loading time                              hr                    1
                         Number of reactors                                             11
                         Residence time                            hr                   60
                         Vessel type                                              Vertical
                         Length                                     m                 21.9
                         Diameter                                   m                 7.29
                         Weight                                    kg             1.12e+05
                         Wall thickness                             m               0.0119
                         Jacketed diameter                          m                  7.4
                         Vessel material                               Stainless steel 316
Purchase cost            Vertical pressure vessel (jacket...      USD             4.63e+06
                         Platform and ladders (x11)               USD             9.69e+05
                         Agitator - Agitator (x11)                USD             1.22e+06
Total purchase cost                                               USD             6.82e+06
Installed equipment cost                                          USD             2.15e+07
Utility cost                                                   USD/hr                  281

Note how all BioSTEAM results match our calculations.

### NREL Anaerobic Bioreactor Model

While modeling of mass and energy balances is standard, the design and costing of the anaerobic bioreactors may have different approaches in prelimiary techno-economic analysis. A report from NREL on cellulosic ethanol production offers cost correlations based on reactor volume and duty. Let's use this model and have a look at how results may difer.

In [8]:
bioreactor = bst.NRELAnaerobicBatchBioreactor(
    ins=feed, 
    outs=('vent', 'effluent'),
    reactions=fermentation,
    T=T_operation,
    V_wf=f,
    V_max=V_max,
    tau=tau_reaction,
    tau_0=tau_cleanup,
    loading_time=tau_loading,
)
bioreactor.simulate()
bioreactor.show('cwt')

NRELAnaerobicBatchBioreactor: bioreactor
ins...
[0] feed  
    phase: 'l', T: 305.15 K, P: 101325 Pa
    composition (%): Water    80
                     Glucose  20
                     -------  1.5e+05 kg/hr
[1] -  
    phase: 'l', T: 298.15 K, P: 101325 Pa
    flow: 0
outs...
[0] vent  
    phase: 'g', T: 305.15 K, P: 101325 Pa
    composition (%): Water    1.87
                     Ethanol  2.15
                     CO2      96
                     -------  1.05e+04 kg/hr
[1] effluent  
    phase: 'l', T: 305.15 K, P: 101325 Pa
    composition (%): Water    85.9
                     Ethanol  10.3
                     Glucose  1.07
                     CO2      2.78
                     -------  1.4e+05 kg/hr


In [9]:
bioreactor.results()

NRELAnaerobic batch bioreactor                          Units  bioreactor
Electricity              Power                             kW         117
                         Cost                          USD/hr        9.11
Chilled water            Duty                           kJ/hr   -1.43e+07
                         Flow                         kmol/hr     9.6e+03
                         Cost                          USD/hr        71.6
Design                   Reactor volume                    m3         931
                         Batch time                        hr          64
                         Loading time                      hr           1
                         Number of reactors                            11
                         Recirculation flow rate        m3/hr        13.1
                         Reactor duty                   kJ/hr   -1.43e+07
                         Cleaning and unloading time       hr           3
                         Working volume fraction                      0.9
Purchase cost            Heat exchangers (x11)            USD    2.19e+05
                         Reactors (x11)                   USD    5.01e+06
                         Agitators (x11)                  USD    3.11e+05
                         Cleaning in place                USD    1.97e+05
                         Recirculation pumps (x11)        USD    1.37e+05
Total purchase cost                                       USD    5.87e+06
Installed equipment cost                                  USD    9.13e+06
Utility cost                                           USD/hr        80.7

The NREL model has significantly lower capital and operational costs. It is normal for equipment costs to vary this much, but they should be within the same order of magnitude. The best method for estimated capital cost is by industry quotes, but even then there are always unforseen expenses during construction.

## Aerobic bioreactor

### Fermentation performance

We will assume the following fermentation performance:


### Mass balance calculation


### Energy balance calculation


### Bioreactor sizing


### Heat exchanger sizing


### Aerobic Bioreactor Model
